# CSIRO Biomass Prediction - Inference Pipeline (ResNet18 Ensemble)

## 1. Executive Summary
This notebook implements the inference pipeline for our top-performing solution (Private Score: **0.48**, Public Score: **0.52**). 

**Methodology:**
- **Architecture**: ResNet18 (Residual Network with 18 layers).
- **Ensemble Strategy**: A 3-Fold averaging ensemble to reduce variance and improve generalization.
- **Preprocessing**: Target variable standardization (Z-score normalization) and CenterCrop resizing.

## 2. Model Architecture: ResNet18
We utilize **ResNet18** [He et al., 2015](https://arxiv.org/abs/1512.03385) as our backbone. Despite being a relatively shallow architecture compared to DenseNet or EfficientNet, ResNet18 proved most robust for this biomass estimation task. 

**Key Advantages for this Task:**
- **Generalization**: With limited training samples, deeper models (like DenseNet121) carried a higher risk of overfitting. ResNet18 provides a balance of capacity and regularization.
- **Residual Connections**: These allow gradients to flow through the network more easily, mitigating the vanishing gradient problem and facilitating the learning of identity mappings.

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Computation Device:", device)

## 3. Configuration & Normalization
We load pre-computed normalization statistics (`outer_mean.npy`, `outer_std.npy`) derived from the training set targets. This ensures that our predictions can be correctly mapped back to the original biomass (grams) scale.

> **Note**: Regressing on normalized targets ($z = \frac{x - \mu}{\sigma}$) stabilizes the training loss, especially when different targets have vastly different magnitudes (e.g., `Dry_Total_g` vs `Dry_Clover_g`).

In [ ]:
# Paths
TEST_CSV = '/kaggle/input/csiro-biomass/test.csv'
TEST_IMG_DIR = '/kaggle/input/csiro-biomass/test' 
WEIGHTS_DIR = '/kaggle/input/cisro-resnet18-submission/'

# Load Normalization Statistics
outer_mean = np.load(WEIGHTS_DIR + 'outer_mean.npy')
outer_std = np.load(WEIGHTS_DIR + 'outer_std.npy')
target_columns = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

## 4. Model Definition
We redefine the ResNet18 structure exactly as it was during training, replacing the final fully connected layer to output our 5 target variables.

In [ ]:
def create_model_RN18():
    model = models.resnet18(weights=None) 
    model.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 5)
    )
    return model

## 5. Data Pipeline
### Preprocessing Strategy
During training, we observed that `CenterCrop(224)` after resizing to `256` yielded better results than direct resizing. This preserves the aspect ratio of the foliage features, which is critical for biomass density estimation.

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        img = Image.open(os.path.join(self.img_dir, img_path)).convert('RGB')
        return self.transform(img), img_path

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),        
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

## 6. Ensemble Inference
We load weights from 3 different folds (`fold_0`, `fold_1`, `fold_2`). By averaging their predictions, we mitigate the risk of a single model having learnt fold-specific biases.

$$ \hat{y}_{final} = \frac{1}{N} \sum_{i=0}^{N} M_i(x) $$

In [ ]:
# Load Data
df_test = pd.read_csv(TEST_CSV)
df_test['image_path'] = df_test['image_path'].str.replace(r'^test/', '', regex=True)
df_unique = df_test[['image_path']].drop_duplicates().reset_index(drop=True)
print(f"Predicting on {len(df_unique)} unique images")

# Load Ensemble Models
model_paths = [f"{WEIGHTS_DIR}best_model_fold_test_{i}.pth" for i in range(3)]
models_list = []

for path in model_paths:
    model = create_model_RN18()
    # Load weights safely
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device).eval()
    models_list.append(model)

# Run Inference
test_ds = TestDataset(df_unique, TEST_IMG_DIR, eval_transform)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

all_img_paths = []
all_preds = []

with torch.no_grad():
    for imgs, paths in test_dl:
        # Ensemble Step: Stack predictions and take mean
        preds = torch.stack([m(imgs.to(device)) for m in models_list]).mean(0)
        all_preds.append(preds.cpu())
        all_img_paths.extend(paths)

# Denormalize Predictions
preds = torch.cat(all_preds, dim=0).numpy()
preds_orig = preds * outer_std + outer_mean

# Save Submission
rows = []
for i, img_path in enumerate(all_img_paths):
    img_id = os.path.splitext(os.path.basename(img_path))[0]
    for j, target in enumerate(target_columns):
        sample_id = f"{img_id}__{target}"
        rows.append({"sample_id": sample_id, "target": preds_orig[i, j]})

submission = pd.DataFrame(rows)[['sample_id', 'target']]
submission.to_csv('submission.csv', index=False)

print("✅ Submission saved successfully.")
print(submission.head())